# WorkON Support Tickets Analysis
## Analysis of Support Tickets from November 2025 to June 2026

This notebook analyzes support ticket data from the WorkON team to identify ticket types, monthly distribution, and average resolution times.

## 1. Load and Explore the Data

In [7]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load the data with correct header row
file_path = '/Users/dudd/Downloads/my_smt/raw.xlsx'
df = pd.read_excel(file_path, header=12)

# Display basic information
print("Dataset Shape:", df.shape)
print("\n" + "="*80)
print("Column Names:")
print("="*80)
for i, col in enumerate(df.columns):
    print(f"{i+1}. {col}")
print("\n" + "="*80)
print("Data Types:")
print("="*80)
print(df.dtypes)
print("\n" + "="*80)
print("First 3 rows:")
print("="*80)
df.head(3)

Dataset Shape: (1616, 49)

Column Names:
1. Incident ID
2. Original Incident Number
3. Requisition ID
4. Created Date (UTC+0)
5. Open Incident Type
6. Current Incident Type
7. Status
8. Status Reason
9. Company
10. Customer Department
11. Region
12. Site Group
13. Site
14. Desk Location
15. Reported Source
16. Summary
17. Impact
18. Open Priority
19. Current Priority
20. Assigned Group
21. Assigned Group Department
22. In Progess Time (hhh:mi)

23. Actual Duration/Open Time (hhh:mi)
24. Down Time of CI-Unavailability (hhh:mi)
25. Last Modified Date (UTC+0)
26. First Resolved Date (UTC+0)
27. Last Resolved Date (UTC+0)
28. Created Month
29. Operational Categorization Tier 1
30. Operational Categorization Tier 2
31. Operational Categorization Tier 3
32. Service+
33. CI+
34. Product Categorization Tier 1
35. Product Categorization Tier 2
36. Product Categorization Tier 3
37. Product Name
38. Resolution Categorization Tier 1
39. Resolution Categorization Tier 2
40. Resolution Categorizatio

,Incident ID,Original Incident Number,Requisition ID,Created Date (UTC+0),Open Incident Type,Current Incident Type,Status,Status Reason,Company,Customer Department,...,Resolution Categorization Tier 3,Resolution Product Categorization Tier 1,Resolution Product Categorization Tier 2,Resolution Product Categorization Tier 3,Resolution Product Name,Primary Center Code,Target Date,Notes,Resolution,Created by CI-Hotline
0,INC000030775471,INC000031253791,NaN,10.11.2025 00:20,User Service Restoration,User Service Restoration,Closed,Resolved by Original Incident,PT,HzP/TEF1-A,...,- None -,Service,Commercial Service,Enterprise Application Service,NaN,560121,,I: ITSD: WorkON is not a business critical ser...,"Dear Customer, Description: cannot access int...",Yes
1,INC000030775473,INC000031153249,NaN,10.11.2025 00:36,User Service Restoration,User Service Restoration,Closed,Resolved by Original Incident,PT,PT-GT/ETS5-Hz,...,- None -,Service,Commercial Service,Enterprise Application Service,NaN,560940,,I: ITSD: WorkON is not a business critical ser...,Dear Customer Description: Customers are cur...,Yes
2,INC000030775478,INC000031153249,NaN,10.11.2025 00:42,User Service Restoration,User Service Restoration,Closed,Resolved by Original Incident,RBJP,RBJP/OFE-P,...,- None -,Service,Commercial Service,Enterprise Application Service,NaN,28000,,I: ITSD: WorkON is not a business critical ser...,Dear Customer Description: Customers are cur...,No


In [8]:
# Parse dates and calculate resolution time FIRST (before using any date columns)
print("="*80)
print("DATA CLEANING & DATE PARSING")
print("="*80)

# Convert date columns
df['Created Date'] = pd.to_datetime(df['Created Date (UTC+0)'], format='%d.%m.%Y %H:%M', errors='coerce')
df['First Resolved Date'] = pd.to_datetime(df['First Resolved Date (UTC+0)'], format='%d.%m.%Y %H:%M', errors='coerce')

# Calculate resolution time in hours and days
df['Resolution Time (hours)'] = (df['First Resolved Date'] - df['Created Date']).dt.total_seconds() / 3600
df['Resolution Time (days)'] = df['Resolution Time (hours)'] / 24

# Extract month-year from created date (IMPORTANT: must be before using in other cells)
df['Created Month-Year'] = df['Created Date'].dt.to_period('M')
df['Resolved Month-Year'] = df['First Resolved Date'].dt.to_period('M')
df['Created Month'] = df['Created Date'].dt.strftime('%Y-%m')

print(f"✓ Date parsing complete")
print(f"✓ Total tickets: {len(df)}")
print(f"✓ Tickets with resolution time: {df['Resolution Time (hours)'].notna().sum()}")
print(f"✓ Average resolution time: {df['Resolution Time (hours)'].mean():.1f} hours ({df['Resolution Time (days)'].mean():.1f} days)")
print(f"✓ Date range: {df['Created Date'].min()} to {df['Created Date'].max()}")

# Show sample
print("\nSample data:")
print(df[['Incident ID', 'Summary', 'Created Date', 'First Resolved Date', 'Resolution Time (days)', 'Created Month-Year']].head(3))

DATA CLEANING & DATE PARSING
✓ Date parsing complete
✓ Total tickets: 1616
✓ Tickets with resolution time: 1616
✓ Average resolution time: 349.9 hours (14.6 days)
✓ Date range: 2025-11-09 23:52:00 to 2026-05-29 05:53:00

Sample data:
       Incident ID                                            Summary  \
0  INC000030775471  Unable to properly approve the workflow for wo...   
1  INC000030775473                     user cannot approve in workon#   
2  INC000030775478                              WorkOn error message#   

         Created Date First Resolved Date  Resolution Time (days)  \
0 2025-11-10 00:20:00 2026-02-25 03:54:00              107.148611   
1 2025-11-10 00:36:00 2026-02-10 06:04:00               92.227778   
2 2025-11-10 00:42:00 2026-02-10 06:05:00               92.224306   

  Created Month-Year  
0            2025-11  
1            2025-11  
2            2025-11  


In [5]:
# Primary classification: Use Resolution Categorization Tier 1 as main category
print("="*80)
print("TICKET CLASSIFICATION")
print("="*80)

# Use existing categorization
df['Main Category'] = df['Resolution Categorization Tier 1'].fillna('Unknown')
df['Sub Category'] = df['Resolution Categorization Tier 2'].fillna('Unknown')

# Show distribution of main categories
print("\nMain Categories Distribution:")
print("="*80)
main_cat_dist = df['Main Category'].value_counts()
for cat, count in main_cat_dist.items():
    pct = (count / len(df)) * 100
    print(f"{cat}: {count} tickets ({pct:.1f}%)")

print("\n" + "="*80)
print("Sub-Categories Distribution (Top 20):")
print("="*80)
sub_cat_dist = df['Sub Category'].value_counts().head(20)
for cat, count in sub_cat_dist.items():
    pct = (count / len(df)) * 100
    print(f"{cat}: {count} tickets ({pct:.1f}%)")

# Create detailed classification combining multiple columns
print("\n" + "="*80)
print("Detailed Ticket Types (Main Category + Sub Category):")
print("="*80)
df['Ticket Type'] = df['Main Category'] + ' - ' + df['Sub Category']
ticket_type_dist = df['Ticket Type'].value_counts().head(30)
print(ticket_type_dist)

# Show sample resolutions for each category
print("\n" + "="*80)
print("Sample Resolutions by Main Category:")
print("="*80)
for cat in df['Main Category'].unique()[:3]:
    print(f"\n--- {cat} ---")
    sample = df[df['Main Category'] == cat]['Resolution'].dropna().head(1).values
    if len(sample) > 0:
        resolution_text = str(sample[0])[:300] + "..." if len(str(sample[0])) > 300 else str(sample[0])
        print(resolution_text)

TICKET CLASSIFICATION

Main Categories Distribution:
Request: 955 tickets (59.1%)
Application: 345 tickets (21.3%)
System: 316 tickets (19.6%)

Sub-Categories Distribution (Top 20):
- None -: 1189 tickets (73.6%)
Outage: 280 tickets (17.3%)
Consulting: 81 tickets (5.0%)
Performance: 35 tickets (2.2%)
User Rights: 25 tickets (1.5%)
Failure: 3 tickets (0.2%)
Documentation: 2 tickets (0.1%)
Backup: 1 tickets (0.1%)

Detailed Ticket Types (Main Category + Sub Category):
Ticket Type
Request - - None -         846
Application - - None -     342
System - Outage            280
Request - Consulting        81
System - Performance        35
Request - User Rights       25
Application - Failure        3
Request - Documentation      2
Request - Backup             1
System - - None -            1
Name: count, dtype: int64

Sample Resolutions by Main Category:

--- System ---
Dear Customer,  Description: cannot access into WORKON, shows error Reason: There was a slowness issue in EMEA P2 Server. Solut

In [10]:
print("="*100)
print("PHÂN LOẠI TICKET - CHI TIẾT (16 LOẠI CỤ THỂ - KHÔNG CÓ 'OTHER')")
print("="*100)

# Read the final classification file
classification_file = '/Users/dudd/Downloads/my_smt/KẾT_QUẢ_PHÂN_LOẠI_FINAL.xlsx'

try:
    # Read sheets
    dist_df = pd.read_excel(classification_file, sheet_name='Phân Loại')
    all_tickets_df = pd.read_excel(classification_file, sheet_name='Tất Cả Tickets (1616)')
    
    print("\n📊 THỐNG KÊ PHÂN LOẠI")
    print("="*100)
    print(dist_df.to_string(index=False))
    
    print(f"\n\n✅ Phân loại chi tiết thành công!")
    print(f"   Tổng tickets: {len(all_tickets_df)}")
    print(f"   Số loại: {len(dist_df)}")
    print(f"   Thời gian resolve TB: {all_tickets_df['Thời Gian (ngày)'].mean():.1f} ngày")
    
except FileNotFoundError:
    print(f"❌ File not found: {classification_file}")
except Exception as e:
    print(f"❌ Error: {e}")

PHÂN LOẠI TICKET - CHI TIẾT (16 LOẠI CỤ THỂ - KHÔNG CÓ 'OTHER')

📊 THỐNG KÊ PHÂN LOẠI
                   Phân Loại  Số Lượng Tỷ Lệ (%)  Thời Gian TB (ngày)  Thời Gian Trung Vị (ngày)
         User Access Request       711     44.0%                 17.2                        7.3
   Application Error/Failure       401     24.8%                 12.4                        5.8
         User Profile Update       260     16.1%                 11.3                        5.2
         Form/Workflow Issue       139      8.6%                 14.7                        5.9
Configuration/Change Request        24      1.5%                  9.5                        4.8
  User Role/Group Assignment        22      1.4%                 17.3                        5.8
         Consulting/Advisory        10      0.6%                 29.3                       14.5
   System Migration/Transfer         9      0.6%                 17.6                       12.9
       Server/Infrastructure         8   